# Предрасчёт эмбеддингов SciFact для Advanced RAG

Этот вспомогательный ноутбук подготавливает только основную коллекцию SciFact, которую использует `vector_search.ipynb`. Он скачивает корпус, запросы и qrels с Hugging Face, кодирует документы и запросы тремя embedding-моделями и сохраняет готовые матрицы в `data/embeddings/scifact/`.

Во время семинара модели не скачиваются и эмбеддинги не вычисляются: основной ноутбук загружает сохранённые `.npy`-файлы.

## Источники на Hugging Face

- Коллекция: [`mteb/scifact`](https://huggingface.co/datasets/mteb/scifact)
- Модель [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- Модель [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5)
- Модель [`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small)


## Установка окружения

Запускайте ноутбук из каталога `ozon-llm-1-course/seminars/05-rag-advanced` в общем окружении из `requirements.txt`

In [1]:
from pathlib import Path
import gc
import json
import platform
import sys
from time import perf_counter

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("platform:", platform.platform())


/Users/ivabystrovd/projects/rag_seminar/.venv/faiss311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 17:50:58.978000 48297 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


python: 3.11.15
torch: 2.15.0
numpy: 2.4.6
platform: macOS-26.6.2-arm64-arm-64bit


In [2]:
# Ноутбук запускается из каталога 05-rag-advanced.
SEMINAR_DIR = Path.cwd()
EMBEDDINGS_DIR = SEMINAR_DIR / "data" / "embeddings"
SCIFACT_DIR = EMBEDDINGS_DIR / "scifact"
SCIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "mteb/scifact"
DATASET_URL = "https://huggingface.co/datasets/mteb/scifact"

print("seminar directory:", SEMINAR_DIR)
print("SciFact artifacts:", SCIFACT_DIR)


seminar directory: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced
SciFact artifacts: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced/data/embeddings/scifact


## Конфигурация embedding-моделей

Все матрицы сохраняются как нормализованные `float32`. Поэтому в `vector_search.ipynb` inner product совпадает с cosine similarity.

Для E5 используются рекомендованные префиксы `query:` и `passage:`. Для BGE добавляется инструкция к запросу; документы остаются без префикса.

In [3]:
MODEL_SPECS = {
    "all_minilm_l6_v2": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "url": "https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2",
        "family": "english",
        "query_prefix": "",
        "passage_prefix": "",
    },
    "bge_small_en_v1_5": {
        "name": "BAAI/bge-small-en-v1.5",
        "url": "https://huggingface.co/BAAI/bge-small-en-v1.5",
        "family": "english",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
    },
    "multilingual_e5_small": {
        "name": "intfloat/multilingual-e5-small",
        "url": "https://huggingface.co/intfloat/multilingual-e5-small",
        "family": "multilingual",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
}

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

BATCH_SIZE = 64 if DEVICE != "cpu" else 32
print("device:", DEVICE, "batch_size:", BATCH_SIZE)
for slug, spec in MODEL_SPECS.items():
    print(f"{slug}: {spec['name']}")
    print(f"  {spec['url']}")


device: cpu batch_size: 32
all_minilm_l6_v2: sentence-transformers/all-MiniLM-L6-v2
  https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
bge_small_en_v1_5: BAAI/bge-small-en-v1.5
  https://huggingface.co/BAAI/bge-small-en-v1.5
multilingual_e5_small: intfloat/multilingual-e5-small
  https://huggingface.co/intfloat/multilingual-e5-small


## Загрузка SciFact

В датасете Hugging Face используются отдельные конфигурации:

- `corpus` — 5 183 научные аннотации;
- `queries` — полный набор запросов;
- default qrels — две разметки: `train` и `test`.

Обе qrels-разметки содержат оценки релевантности. Для нашего стенда объединяем положительные связи из `train` и `test`, чтобы получить больше запросов и более стабильные средние метрики. Имя файла `queries_test.jsonl` ниже сохраняем ради совместимости с уже написанным `vector_search.ipynb`: теперь в нём лежит объединённая выборка, а не только test.

In [4]:
scifact_corpus = load_dataset(DATASET_NAME, "corpus", split="corpus")
scifact_queries_all = load_dataset(DATASET_NAME, "queries", split="queries")
scifact_qrels_train = load_dataset(DATASET_NAME, split="train")
scifact_qrels_test = load_dataset(DATASET_NAME, split="test")

print("dataset:", DATASET_NAME)
print("dataset page:", DATASET_URL)
print("corpus rows:", len(scifact_corpus))
print("all query rows:", len(scifact_queries_all))
print("train qrels rows:", len(scifact_qrels_train))
print("test qrels rows:", len(scifact_qrels_test))
print("corpus example:", scifact_corpus[0])
print("query example:", scifact_queries_all[0])
print("train qrel example:", scifact_qrels_train[0])
print("test qrel example:", scifact_qrels_test[0])

dataset: mteb/scifact
dataset page: https://huggingface.co/datasets/mteb/scifact
corpus rows: 5183
all query rows: 1109
train qrels rows: 919
test qrels rows: 339
corpus example: {'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central whit

In [5]:
def sort_key(value):
    value = str(value)
    return (0, int(value)) if value.isdigit() else (1, value)

scifact_corpus_rows = [dict(row) for row in scifact_corpus]
scifact_queries_rows = [dict(row) for row in scifact_queries_all]
scifact_qrels_rows = [
    *[dict(row) for row in scifact_qrels_train],
    *[dict(row) for row in scifact_qrels_test],
]

scifact_query_ids = sorted(
    {str(row["query-id"]) for row in scifact_qrels_rows}, key=sort_key
)
scifact_query_by_id = {str(row["_id"]): row for row in scifact_queries_rows}
scifact_queries_with_qrels = [
    scifact_query_by_id[query_id] for query_id in scifact_query_ids
]

scifact_qrels = {query_id: [] for query_id in scifact_query_ids}
for row in scifact_qrels_rows:
    if float(row["score"]) > 0:
        query_id = str(row["query-id"])
        corpus_id = str(row["corpus-id"])
        if corpus_id not in scifact_qrels[query_id]:
            scifact_qrels[query_id].append(corpus_id)

print("queries with train+test qrels:", len(scifact_queries_with_qrels))
print("queries with at least one positive:", sum(bool(v) for v in scifact_qrels.values()))
print("first query:", scifact_queries_with_qrels[0])
print("first qrels:", scifact_qrels[scifact_query_ids[0]])

queries with train+test qrels: 1109
queries with at least one positive: 1109
first query: {'_id': '0', 'text': '0-dimensional biomaterials lack inductive properties.'}
first qrels: ['31715818']


## Сохранение текстов и qrels

Индекс хранит только плотные векторы и integer-позиции. Исходные тексты, внешние IDs и qrels сохраняем рядом, чтобы `vector_search.ipynb` мог восстановить найденные документы и оценить qrels recall.

In [6]:
def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


def corpus_text(row):
    title = str(row.get("title", "")).strip()
    text = str(row.get("text", "")).strip()
    return f"{title}\n{text}" if title else text


write_jsonl(SCIFACT_DIR / "corpus.jsonl", scifact_corpus_rows)
# Имена оставляем прежними, чтобы vector_search.ipynb не требовал изменений.
write_jsonl(SCIFACT_DIR / "queries_test.jsonl", scifact_queries_with_qrels)
(SCIFACT_DIR / "qrels_test.json").write_text(
    json.dumps(scifact_qrels, ensure_ascii=False, indent=2), encoding="utf-8"
)

scifact_doc_texts = [corpus_text(row) for row in scifact_corpus_rows]
scifact_query_texts = [corpus_text(row) for row in scifact_queries_with_qrels]

print("saved metadata to:", SCIFACT_DIR)
print("documents:", len(scifact_doc_texts))
print("queries with train+test qrels:", len(scifact_query_texts))

saved metadata to: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced/data/embeddings/scifact
documents: 5183
queries with train+test qrels: 1109


## Кодирование и сохранение матриц

Для каждой модели сохраняются:

- `docs.npy` — нормализованные эмбеддинги всех документов;
- `queries.npy` — нормализованные эмбеддинги запросов с train+test qrels;
- `metadata.json` — модель, ссылка на карточку, размерность и параметры кодирования.

Расчёт документов и запросов оставлен отдельными вызовами, чтобы было видно, где применяются разные префиксы.

In [7]:
def prefixed_texts(texts, prefix):
    return [prefix + text for text in texts]


def encode_and_save(model, model_slug, model_spec):
    output_dir = SCIFACT_DIR / model_slug
    output_dir.mkdir(parents=True, exist_ok=True)

    started = perf_counter()
    docs_matrix = model.encode(
        prefixed_texts(scifact_doc_texts, model_spec["passage_prefix"]),
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype("float32", copy=False)
    queries_matrix = model.encode(
        prefixed_texts(scifact_query_texts, model_spec["query_prefix"]),
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype("float32", copy=False)

    np.save(output_dir / "docs.npy", docs_matrix)
    np.save(output_dir / "queries.npy", queries_matrix)
    metadata = {
        "dataset": "scifact",
        "dataset_name": DATASET_NAME,
        "dataset_url": DATASET_URL,
        "qrels_splits": ["train", "test"],
        "query_selection": "all query IDs present in train or test qrels",
        "model_slug": model_slug,
        "model_name": model_spec["name"],
        "model_url": model_spec["url"],
        "family": model_spec["family"],
        "device": DEVICE,
        "dtype": "float32",
        "normalized": True,
        "dimension": int(docs_matrix.shape[1]),
        "documents": int(docs_matrix.shape[0]),
        "queries": int(queries_matrix.shape[0]),
        "elapsed_seconds": perf_counter() - started,
        "query_prefix": model_spec["query_prefix"],
        "passage_prefix": model_spec["passage_prefix"],
    }
    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    print("saved:", output_dir)
    print("  docs:", docs_matrix.shape, docs_matrix.dtype)
    print("  queries:", queries_matrix.shape, queries_matrix.dtype)
    print("  mean norms:", round(float(np.linalg.norm(docs_matrix, axis=1).mean()), 5), round(float(np.linalg.norm(queries_matrix, axis=1).mean()), 5))
    print("  elapsed seconds:", round(metadata["elapsed_seconds"], 1))
    return metadata


def release_model(model):
    del model
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()


## `all-MiniLM-L6-v2`

Компактная англоязычная модель для быстрого baseline на SciFact: [карточка модели на Hugging Face](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

In [8]:
model = SentenceTransformer(MODEL_SPECS["all_minilm_l6_v2"]["name"], device=DEVICE)
minilm_metadata = encode_and_save(
    model,
    "all_minilm_l6_v2",
    MODEL_SPECS["all_minilm_l6_v2"],
)
release_model(model)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10156.89it/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   1%|          | 1/162 [00:00<00:36,  4.42it/s]

Batches:   1%|          | 2/162 [00:00<00:25,  6.21it/s]

Batches:   2%|▏         | 3/162 [00:00<00:22,  7.19it/s]

... (157 outputs left out)

Batches:  74%|███████▍  | 26/35 [00:00<00:00, 63.58it/s]

Batches: 100%|██████████| 35/35 [00:00<00:00, 71.17it/s]

Batches: 100%|██████████| 35/35 [00:00<00:00, 64.14it/s]

saved: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced/data/embeddings/scifact/all_minilm_l6_v2
  docs: (5183, 384) float32
  queries: (1109, 384) float32
  mean norms: 1.0 1.0
  elapsed seconds: 18.3


## `bge-small-en-v1.5`

Основная модель, которую по умолчанию загружает `vector_search.ipynb`: [карточка модели на Hugging Face](https://huggingface.co/BAAI/bge-small-en-v1.5). Для запросов используется инструкция BGE.

In [9]:
model = SentenceTransformer(MODEL_SPECS["bge_small_en_v1_5"]["name"], device=DEVICE)
bge_metadata = encode_and_save(
    model,
    "bge_small_en_v1_5",
    MODEL_SPECS["bge_small_en_v1_5"],
)
release_model(model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10182.59it/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   1%|          | 1/162 [00:00<01:53,  1.42it/s]

Batches:   1%|          | 2/162 [00:01<01:34,  1.69it/s]

Batches:   2%|▏         | 3/162 [00:01<01:27,  1.81it/s]

... (169 outputs left out)

Batches:  80%|████████  | 28/35 [00:01<00:00, 30.36it/s]

Batches:  91%|█████████▏| 32/35 [00:01<00:00, 32.44it/s]

Batches: 100%|██████████| 35/35 [00:01<00:00, 28.57it/s]

saved: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced/data/embeddings/scifact/bge_small_en_v1_5
  docs: (5183, 384) float32
  queries: (1109, 384) float32
  mean norms: 1.0 1.0
  elapsed seconds: 66.0


## `multilingual-e5-small`

Мультиязычный вариант для сравнения embedding-моделей на той же английской SciFact-коллекции: [карточка модели на Hugging Face](https://huggingface.co/intfloat/multilingual-e5-small). Несмотря на название модели, здесь кодируем только SciFact.

In [10]:
model = SentenceTransformer(MODEL_SPECS["multilingual_e5_small"]["name"], device=DEVICE)
e5_metadata = encode_and_save(
    model,
    "multilingual_e5_small",
    MODEL_SPECS["multilingual_e5_small"],
)
release_model(model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9501.04it/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   1%|          | 1/162 [00:00<02:06,  1.27it/s]

Batches:   1%|          | 2/162 [00:01<01:43,  1.55it/s]

Batches:   2%|▏         | 3/162 [00:01<01:33,  1.69it/s]

... (168 outputs left out)

Batches:  77%|███████▋  | 27/35 [00:00<00:00, 32.86it/s]

Batches:  91%|█████████▏| 32/35 [00:01<00:00, 35.52it/s]

Batches: 100%|██████████| 35/35 [00:01<00:00, 30.98it/s]

saved: /Users/ivabystrovd/projects/rag_seminar/ozon-llm-1-course/seminars/05-rag-advanced/data/embeddings/scifact/multilingual_e5_small
  docs: (5183, 384) float32
  queries: (1109, 384) float32
  mean norms: 1.0 1.0
  elapsed seconds: 68.8


## Проверка артефактов и manifest

Последняя ячейка ничего не кодирует. Она проверяет три SciFact-пары, близость норм к единице и записывает краткий `manifest.json`. В `queries_test.jsonl` и `qrels_test.json` к этому моменту уже сохранено объединение train+test.

In [11]:
expected_models = [
    "all_minilm_l6_v2",
    "bge_small_en_v1_5",
    "multilingual_e5_small",
]

manifest = []
for model_slug in expected_models:
    artifact_dir = SCIFACT_DIR / model_slug
    docs_path = artifact_dir / "docs.npy"
    queries_path = artifact_dir / "queries.npy"
    metadata_path = artifact_dir / "metadata.json"
    if not (docs_path.exists() and queries_path.exists() and metadata_path.exists()):
        raise FileNotFoundError(f"Не хватает артефактов для {model_slug}: {artifact_dir}")

    docs_matrix = np.load(docs_path, mmap_mode="r")
    queries_matrix = np.load(queries_path, mmap_mode="r")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    max_norm_error = max(
        float(np.abs(np.linalg.norm(np.asarray(docs_matrix), axis=1) - 1).max()),
        float(np.abs(np.linalg.norm(np.asarray(queries_matrix), axis=1) - 1).max()),
    )
    row = {
        "dataset": "scifact",
        "model_slug": model_slug,
        "model_name": metadata["model_name"],
        "docs_shape": list(docs_matrix.shape),
        "queries_shape": list(queries_matrix.shape),
        "dimension": metadata["dimension"],
        "max_norm_error": max_norm_error,
        "docs_file_mib": round(docs_path.stat().st_size / 1024**2, 2),
        "queries_file_mib": round(queries_path.stat().st_size / 1024**2, 2),
    }
    manifest.append(row)
    print(row)

(EMBEDDINGS_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("manifest entries:", len(manifest))


{'dataset': 'scifact', 'model_slug': 'all_minilm_l6_v2', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'docs_shape': [5183, 384], 'queries_shape': [1109, 384], 'dimension': 384, 'max_norm_error': 1.1920928955078125e-07, 'docs_file_mib': 7.59, 'queries_file_mib': 1.62}
{'dataset': 'scifact', 'model_slug': 'bge_small_en_v1_5', 'model_name': 'BAAI/bge-small-en-v1.5', 'docs_shape': [5183, 384], 'queries_shape': [1109, 384], 'dimension': 384, 'max_norm_error': 1.1920928955078125e-07, 'docs_file_mib': 7.59, 'queries_file_mib': 1.62}
{'dataset': 'scifact', 'model_slug': 'multilingual_e5_small', 'model_name': 'intfloat/multilingual-e5-small', 'docs_shape': [5183, 384], 'queries_shape': [1109, 384], 'dimension': 384, 'max_norm_error': 1.1920928955078125e-07, 'docs_file_mib': 7.59, 'queries_file_mib': 1.62}
manifest entries: 3


Готовые артефакты можно загружать в `vector_search.ipynb`. Для смены embedding-модели достаточно изменить `MODEL_SLUG` в первой секции основного ноутбука на один из трёх slug из `manifest.json`.